In [1]:
import torch
import torch.nn as nn
import torch.optim as optim

class AudioResidualLSTM(nn.Module):
    def __init__(self, input_channels=1, hidden_dim=64, num_layers=2, kernel_size=3):
        """
        Args:
            input_channels: 1 for mono, 2 for stereo.
            hidden_dim: The 'memory' size of the LSTM.
            num_layers: How many LSTM layers to stack.
            kernel_size: Used for a pre-processing convolution to help the LSTM.
        """
        super(AudioResidualLSTM, self).__init__()

        self.input_channels = input_channels
        self.hidden_dim = hidden_dim

        # 1. Pre-processing Layer: A small 1D Convolution.
        # This helps the model look at a tiny window of samples
        # rather than just a single isolated point.
        self.pre_conv = nn.Conv1d(
            in_channels=input_channels,
            out_channels=hidden_dim,
            kernel_size=kernel_size,
            padding=kernel_size // 2
        )

        # 2. The Temporal Engine: LSTM
        # batch_first=True means input shape is (Batch, Sequence, Features)
        self.lstm = nn.LSTM(
            input_size=hidden_dim,
            hidden_size=hidden_dim,
            num_layers=num_layers,
            batch_first=True
        )

        # 3. The Projection Head: The FC/Linear Layer
        # This maps the high-dimensional LSTM state back to audio amplitude.
        self.fc = nn.Linear(hidden_dim, input_channels)

    def forward(self, x):
        """
        x: Input tensor of shape (Batch, Sequence, Channels)
           Example: (32, 1024, 1) -> 32 chunks of 1024 samples each.
        """
        # Store the original input for the residual connection
        identity = x

        # --- Step 1: Pre-processing ---
        # Conv1d expects (Batch, Channels, Sequence), so we permute
        x_conv = x.permute(0, 2, 1)
        x_conv = self.pre_conv(x_conv)

        # Bring it back to (Batch, Sequence, Hidden_Dim) for the LSTM
        x_features = x_conv.permute(0, 2, 1)

        # --- Step 2: Temporal Modeling ---
        # lstm_out shape: (Batch, Sequence, Hidden_Dim)
        lstm_out, (h_n, c_n) = self.lstm(x_features)

        # --- Step 3: Projection ---
        # Map hidden features back to audio channels
        # residual shape: (Batch, Sequence, Channels)
        residual = self.fc(lstm_out)

        # --- Step 4: The Residual Connection (CRITICAL) ---
        # We add the learned distortion back to the original clean signal.
        # Output = Clean_Signal + Learned_Distortion
        output = identity + residual

        return output

# --- Testing the Implementation ---

def test_model():
    # Hyperparameters
    batch_size = 16
    seq_len = 512  # Number of audio samples per chunk
    channels = 1   # Mono audio
    hidden_size = 128

    # 1. Initialize Model, Loss, and Optimizer
    model = AudioResidualLSTM(input_channels=channels, hidden_dim=hidden_size)
    criterion = nn.MSELoss() # Mean Squared Error is standard for audio reconstruction
    optimizer = optim.Adam(model.parameters(), lr=0.001)

    # 2. Create Dummy Data
    # 'clean_audio' is our input
    # 'distorted_audio' is our target (the ground truth from the amplifier)
    clean_audio = torch.randn(batch_size, seq_len, channels)
    distorted_audio = clean_audio * torch.randn(batch_size, seq_len, channels).abs() # Randomly distorted

    # 3. Simple Training Loop Demo
    model.train()
    for epoch in range(5):
        optimizer.zero_grad()

        # Forward pass
        predictions = model(clean_audio)

        # Calculate loss (how close is our predicted distortion to the real distortion?)
        loss = criterion(predictions, distorted_audio)

        # Backward pass
        loss.backward()
        optimizer.step()

        print(f"Epoch [{epoch+1}/5], Loss: {loss.item():.6f}")

    print("\nModel training simulation complete.")
    print(f"Output Shape: {predictions.shape}") # Should match input shape

if __name__ == "__main__":
    test_model()


Epoch [1/5], Loss: 0.431191
Epoch [2/5], Loss: 0.424528
Epoch [3/5], Loss: 0.420500
Epoch [4/5], Loss: 0.414349
Epoch [5/5], Loss: 0.409897

Model training simulation complete.
Output Shape: torch.Size([16, 512, 1])
